# 3 From a probability to a decision

The last two notebooks produced *distributions*: a forecast with a spread, a prediction
set with a coverage guarantee. Nobody acts on a distribution. At some point somebody has
to approve the loan, order the stock, flag the scan, or let it through.

That step — turning a number into an action — is where most of the damage in applied
machine learning happens, and it is usually done by accident:

```python
if p > 0.5:
    ...
```

**0.5 is not a property of your model.** It is a decision, silently made on your behalf,
that treats a false positive and a false negative as equally bad. This notebook makes that
choice explicit and measures what it is worth.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from notebooktester import param
from torch import nn
from vectormesh import VectorCache
from vectormesh.components import NeuralNet

TARGET = 0          # one class against the other 46
BATCH = 128
EPOCHS = param(20, test=1)

cache = VectorCache.from_hub("pttrn-io/dtd-dinov2-small")
d = cache.dataset.shuffle(seed=0).with_format("numpy")
X = torch.tensor(np.asarray(d["embed"], dtype="float32"))
y = torch.tensor((np.asarray(d["label"]) == TARGET).astype("int64"))

a, b = int(0.50 * len(X)), int(0.75 * len(X))
Xtr, ytr = X[:a], y[:a]
Xca, yca = X[a:b], y[a:b]     # choose the threshold here
Xte, yte = X[b:], y[b:]       # score it here

print(f"positives: {int(y.sum())} of {len(y)}  ({float(y.float().mean()):.2%})")

A **one-vs-rest** task, which is where thresholds actually get chosen in practice: is this
the thing I am looking for, yes or no. Turning 47 classes into one gives roughly a 2%
positive rate for free — and imbalance like that is the norm in anything worth detecting.
Fraud, disease, equipment failure, and defects are all rare by definition.

In [ ]:
torch.manual_seed(0)
model = NeuralNet(X.shape[1], 2)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
for _ in range(EPOCHS):
    model.train()
    perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr) - 1, BATCH):
        idx = perm[i:i + BATCH]
        opt.zero_grad()
        loss_fn(model(Xtr[idx]), ytr[idx]).backward()
        opt.step()
model.eval()

with torch.no_grad():
    p_ca = torch.softmax(model(Xca), -1)[:, 1].numpy()
    p_te = torch.softmax(model(Xte), -1)[:, 1].numpy()
yt, yc = yte.numpy(), yca.numpy()

print(f"accuracy at threshold 0.5      : {((p_te >= 0.5).astype(int) == yt).mean():.4f}")
print(f"accuracy of always saying 'no' : {(yt == 0).mean():.4f}")

Start with the number nobody should have reported. The model scores about 0.986, which
sounds excellent until you notice that **refusing to ever predict a positive scores about
0.978.** A whole trained model, for less than a point over a constant.

This is not a bad model — it is a bad *metric*. On a 2% base rate, accuracy is dominated
by the majority class and cannot see what you care about. That is the honest version of
the precision/recall discussion: the reason to look past accuracy is not that it is
unfashionable, it is that on imbalanced problems it is nearly constant.

## 3.1 Say what a mistake costs

Two mistakes, two prices:

- a **false positive** costs `C_fp` — you investigated something harmless
- a **false negative** costs `C_fn` — you missed the thing you were looking for

You do not need real currency, only the *ratio*. "Missing one is worth twenty false
alarms" is a sentence a domain expert can say out loud, and it is all the information the
threshold needs.

In [ ]:
def confusion(thr, p, yy):
    pred = p >= thr
    return (int((pred & (yy == 1)).sum()), int((pred & (yy == 0)).sum()),
            int((~pred & (yy == 1)).sum()), int((~pred & (yy == 0)).sum()))   # tp, fp, fn, tn


def expected_cost(thr, c_fp, c_fn, p, yy):
    _, fp, fn, _ = confusion(thr, p, yy)
    return (fp * c_fp + fn * c_fn) / len(yy)


GRID = np.linspace(0.01, 0.99, 99)

print(f"{'C_fp':>5s} {'C_fn':>5s} {'best thr':>9s} {'cost there':>11s} {'cost@0.5':>9s} {'saved':>7s}")
for c_fp, c_fn in [(1, 1), (1, 5), (1, 20), (5, 1)]:
    costs = [expected_cost(t, c_fp, c_fn, p_te, yt) for t in GRID]
    i = int(np.argmin(costs))
    c05 = expected_cost(0.5, c_fp, c_fn, p_te, yt)
    print(f"{c_fp:5d} {c_fn:5d} {GRID[i]:9.2f} {costs[i]:11.4f} {c05:9.4f} "
          f"{100 * (c05 - costs[i]) / c05:6.1f}%")

Read the last column: **choosing the threshold instead of accepting 0.5 removes half to
three quarters of the expected cost.** No new data, no new architecture, no retraining —
the same model, asked the right question.

And read the *first row*, which is the one that should surprise you. Even when both
mistakes cost exactly the same, the best threshold is nowhere near 0.5.

The textbook answer says the optimal threshold is `C_fp / (C_fp + C_fn)` — 0.5 for equal
costs. That formula is correct, and it assumes the model's probabilities are **calibrated**:
that `p = 0.3` really does mean a 30% chance. This model's are not, and on a 2% base rate
they are badly compressed toward zero, so the formula's answer is wrong while the
empirically chosen threshold is right.

Which is the same lesson notebook 02 taught from the other direction. *Do not trust the
number, measure the consequence.* Both conformal prediction and threshold selection work
by calibrating against held-out data rather than by believing the model's output.

In [ ]:
c_fp, c_fn = 1, 20
costs = np.array([expected_cost(t, c_fp, c_fn, p_te, yt) for t in GRID])
best = GRID[int(costs.argmin())]

plt.figure(figsize=(7, 4))
plt.plot(GRID, costs)
plt.axvline(0.5, ls="--", color="grey", label="the default nobody chose")
plt.axvline(best, ls="-", color="tab:red", label=f"cost-optimal ({best:.2f})")
plt.xlabel("threshold"); plt.ylabel("expected cost per item")
plt.title(f"C_fp={c_fp}, C_fn={c_fn}"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

The curve is steep on the left and flat on the right, which is worth noticing: being a
little too *eager* costs you almost nothing, while being a little too cautious is
expensive. When missing things is what hurts, the asymmetry of the cost curve is itself
asymmetric.

## 3.2 What the threshold actually did

In [ ]:
print(f"{'threshold':>10s} {'TP':>4s} {'FP':>4s} {'FN':>4s} {'TN':>5s} {'recall':>8s} {'precision':>10s}")
for t in (0.5, best):
    tp, fp, fn, tn = confusion(t, p_te, yt)
    print(f"{t:10.2f} {tp:4d} {fp:4d} {fn:4d} {tn:5d} "
          f"{tp / max(tp + fn, 1):8.3f} {tp / max(tp + fp, 1):10.3f}")

The trade in plain sight. Lowering the threshold catches nearly every positive — recall
goes from about two thirds to about 95% — and pays for it with roughly three times as many
false alarms, dropping precision from around 0.70 to around 0.51.

Whether that is a good deal is **not a modelling question**. At twenty-to-one it obviously
is. At one-to-one it obviously is not. The model cannot tell you which world you are in,
and no amount of tuning will answer a question you have not asked.

## 3.3 Choose the threshold on data you did not test on

Everything above picked the threshold by looking at the test set, which is the same
mistake as reporting a search's best validation score. The threshold is a fitted
parameter — one number, but fitted — so it needs its own split.

In [ ]:
costs_ca = [expected_cost(t, c_fp, c_fn, p_ca, yc) for t in GRID]
t_star = GRID[int(np.argmin(costs_ca))]

print(f"threshold chosen on calibration : {t_star:.2f}")
print(f"  its cost on test              : {expected_cost(t_star, c_fp, c_fn, p_te, yt):.4f}")
print(f"  best achievable on test       : {costs.min():.4f}  (at {best:.2f})")
print(f"  cost of the 0.5 default       : {expected_cost(0.5, c_fp, c_fn, p_te, yt):.4f}")

The honestly chosen threshold lands on the same cost as the oracle here — one fitted
parameter over a thousand calibration points does not have much room to overfit. That is a
reassuring result rather than a licence: fit ten thresholds on ten segments and the gap
will open.

## 3.4 The third option: refuse

Every decision so far was binary. Real systems usually have a third action: **send it to a
human.** That needs a rule for *which* items, and a third price — `C_review`, what an
expert's attention costs on the same scale as the mistakes.

The rule is the uncertainty you have been measuring all lesson. Items whose probability
sits near the threshold are the ones the model is least sure about, so route a band around
it and let the model act on the rest.

In [ ]:
def hybrid_cost(review, c_rev, thr=None):
    thr = t_star if thr is None else thr
    pred = p_te >= thr
    fp = int((pred & (yt == 0) & ~review).sum())
    fn = int((~pred & (yt == 1) & ~review).sum())
    return (fp * c_fp + fn * c_fn + int(review.sum()) * c_rev) / len(yt)


none = np.zeros(len(yt), bool)
print(f"auto-only, no review: {hybrid_cost(none, 0.0):.4f}\n")

print(f"{'review band':>20s} {'reviewed':>9s} {'C_rev=0.5':>10s} {'C_rev=2':>9s} {'C_rev=8':>9s}")
for lo, hi in param([(0.001, 0.05), (0.001, 0.20), (0.0005, 0.50)], test=[(0.001, 0.05)]):
    rev = (p_te >= lo) & (p_te <= hi)
    row = "".join(f"{hybrid_cost(rev, cr):10.4f}" for cr in (0.5, 2.0, 8.0))
    print(f"[{lo:.4f}, {hi:.2f}]".rjust(20) + f" {rev.mean():8.1%}" + row)

Reviewing about 3% of the items removes roughly a third of the expected cost — **if** an
expert costs 0.5 per item. At a review cost of 2 the same policy is worse than not
reviewing at all, and at 8 it is a disaster.

The break-even sits near `C_review ≈ 1` here: reviewing those ~28 items removes about 28
units of expected cost, so it pays exactly while an expert is cheaper than one false alarm.
Since a miss costs 20 in this setup, **review is worth it only while it costs under a
twentieth of a miss** — a sentence you can take to whoever owns the budget.

Note also that the *narrowest* band wins at every review price. Widening it pulls in items
the model was already getting right, so you pay for opinions you did not need. Abstention
is worth targeting tightly rather than generously.

**This is the payoff of the whole lesson.** A point forecast supports none of it. A
distribution plus a cost structure gives you a policy — act, refuse, escalate — with an
expected cost you can defend in a meeting, and a break-even price for the humans.

## What you now know

- **A threshold is a decision, not a model parameter**, and 0.5 is a decision nobody made
  on purpose.
- **On imbalanced problems accuracy is nearly constant** and cannot see what you care
  about. The trivial baseline here scores 0.978.
- **Costs, not metrics, choose the threshold** — and choosing it well was worth 50-75% of
  the expected cost on the same model.
- **The textbook formula assumes calibration.** When probabilities are not calibrated, fit
  the threshold empirically on held-out data instead.
- **Abstention is a third action** with its own price, and whether it pays is arithmetic.

## Exercises

**E1 — ask for the ratio, not the currency.** Pick a domain you know and state
`C_fn / C_fp` for it in one sentence, the way you would say it to a colleague. Then compute
the threshold it implies here. How far is it from 0.5, and would anyone have guessed?

**E2 — does calibration rescue the formula?** Fit a single temperature on the calibration
split so the probabilities are better calibrated, then compare the theoretical threshold
`C_fp/(C_fp+C_fn)` with the empirically optimal one. Does the gap close? This is the
cleanest way to see what calibration actually buys.

**E3 — a rarer positive.** Redo the notebook targeting a class you first make rarer by
subsampling positives to 0.5%. What happens to the optimal threshold, and to the gap
between the model and the always-say-no baseline?

**E4 — the newsvendor.** Notebook 01's quantile head predicts a distribution over demand.
If unsold stock costs `C_o` and a stockout costs `C_u`, the optimal order is the
`C_u / (C_u + C_o)` quantile of that distribution — the same critical-fractile logic as
this notebook, for a continuous decision. Implement it against 01's forecast and compare
with ordering the mean.

**E5 — segment the threshold.** Fit a separate threshold per group (say, textures the
model finds easy versus hard, as in notebook 02 §2.5). Does per-segment thresholding beat
one global threshold on test, or does it overfit the calibration split? Report both.